# Stock Price Prediction with LSTM (PyTorch)
### Daily Challenge — Week 12

**Goal:**
- Build a preprocessed dataset for stock price prediction
- Train an LSTM model with PyTorch to predict the next-day closing price

**Pipeline:**
1. Install & import libraries
2. Load and explore the dataset
3. Feature engineering
4. Train / Val / Test split + normalize (no leakage)
5. Create PyTorch Dataset / DataLoader
6. Define the LSTM model
7. Train the model
8. Evaluate (R² score) & save artefacts

## Step 1 — Install & Import Libraries

In [ ]:
# !pip install kagglehub torch scikit-learn pandas numpy matplotlib

import os, math, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score

import kagglehub

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
print(f'PyTorch: {torch.__version__}')

## Step 2 — Load and Explore the Dataset

In [ ]:
path = kagglehub.dataset_download("jacksoncrow/stock-market-dataset")
print("Dataset path:", path)

In [ ]:
TICKER     = 'AAPL'
stock_path = os.path.join(path, 'stocks', f'{TICKER}.csv')

df = pd.read_csv(stock_path)
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

print(f'Shape      : {df.shape}')
print(f'Date range : {df["Date"].min().date()} → {df["Date"].max().date()}')
df.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))

axes[0].plot(df['Date'], df['Close'], color='steelblue', linewidth=1)
axes[0].set_title(f'{TICKER} — Raw Close Price (non-stationary)', fontweight='bold')
axes[0].set_ylabel('USD')

daily_return = df['Close'].pct_change().dropna()
axes[1].plot(df['Date'].iloc[1:], daily_return, color='tomato', linewidth=0.6, alpha=0.7)
axes[1].set_title(f'{TICKER} — Daily Return (stationary)', fontweight='bold')
axes[1].set_ylabel('Return')

plt.tight_layout()
plt.show()

print("\nThe raw price grew from $0.40 to $180+ — this is why raw price prediction fails:")
print("a scaler fitted on old prices cannot represent test prices.")
print("Solution: use scale-invariant features + predict a RATIO instead of a raw price.")

## Step 3 — Feature Engineering

All input features are **scale-invariant** (ratios / returns), so they have the same
meaning across different price levels.

The **target** is the *next-day price ratio*:
$$\text{Target\_ratio} = \frac{\text{Close}_{t+1}}{\text{Close}_t} \approx 1.0 \pm 0.05$$
This is always stationary. After inference we reconstruct the USD price:
$$\hat{\text{Close}}_{t+1} = \text{Close}_t \times \hat{\text{ratio}}$$

In [ ]:
df = df[['Date', 'Open', 'High', 'Low', 'Close', 'Volume']].dropna().copy()

# ── Scale-invariant features ─────────────────────────────────────────────────
df['Return']    = df['Close'].pct_change()                          # daily return
df['MA5']       = df['Close'].rolling(5).mean()
df['MA20']      = df['Close'].rolling(20).mean()
df['Ratio_5']   = df['Close'] / df['MA5']                           # price vs 5-day MA
df['Ratio_20']  = df['Close'] / df['MA20']                          # price vs 20-day MA
df['HL_range']  = (df['High'] - df['Low'])   / df['Close']          # intraday range %
df['OC_gap']    = (df['Close'] - df['Open']) / df['Open']           # open-close gap %
df['Vol_ratio'] = df['Volume'] / df['Volume'].rolling(20).mean()    # volume vs 20-day MA

# ── Targets ──────────────────────────────────────────────────────────────────
# What the model learns to predict (stationary)
df['Target_ratio'] = df['Close'].shift(-1) / df['Close']
# What we use for final evaluation in USD (stored separately)
df['Target_price'] = df['Close'].shift(-1)

df = df.dropna().reset_index(drop=True)

FEATURE_COLS = ['Return', 'Ratio_5', 'Ratio_20', 'HL_range', 'OC_gap', 'Vol_ratio']

print(f'Dataset shape : {df.shape}')
print(f'\nTarget_ratio stats (should be centred ~1.0):')
print(df['Target_ratio'].describe().round(5))

## Step 4 — Train / Val / Test Split + Normalize (no leakage)

> **Rule:** `MinMaxScaler` is fitted **only on the training split**, then `.transform()` is
> applied to val and test.  
> Fitting on the full dataset leaks future statistics into the past.

In [ ]:
SEQ_LEN    = 30
BATCH_SIZE = 64

N         = len(df)
train_end = int(N * 0.70)
val_end   = int(N * 0.85)

train_df = df.iloc[:train_end].copy()
val_df   = df.iloc[train_end:val_end].copy()
test_df  = df.iloc[val_end:].copy()

print(f'Train : {len(train_df):,} rows  ({train_df["Date"].iloc[0].date()} → {train_df["Date"].iloc[-1].date()})')
print(f'Val   : {len(val_df):,}  rows  ({val_df["Date"].iloc[0].date()} → {val_df["Date"].iloc[-1].date()})')
print(f'Test  : {len(test_df):,} rows  ({test_df["Date"].iloc[0].date()} → {test_df["Date"].iloc[-1].date()})')

In [ ]:
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

# Fit ONLY on training data
X_train_sc = scaler_X.fit_transform(train_df[FEATURE_COLS].values)
y_train_sc = scaler_y.fit_transform(train_df[['Target_ratio']].values)

# Transform (no fit) on val and test
X_val_sc  = scaler_X.transform(val_df[FEATURE_COLS].values)
y_val_sc  = scaler_y.transform(val_df[['Target_ratio']].values)

X_test_sc = scaler_X.transform(test_df[FEATURE_COLS].values)
y_test_sc = scaler_y.transform(test_df[['Target_ratio']].values)

print(f'Target_ratio range in train : [{train_df["Target_ratio"].min():.4f}, {train_df["Target_ratio"].max():.4f}]')
print(f'Target_ratio range in test  : [{test_df["Target_ratio"].min():.4f}, {test_df["Target_ratio"].max():.4f}]')
print("\nBoth ranges are similar → scaler works correctly on test data.")

## Step 5 — Create PyTorch Dataset / DataLoader

In [ ]:
def build_sequences(X: np.ndarray, y: np.ndarray, seq_len: int):
    """Sliding-window sequences of length seq_len."""
    xs, ys = [], []
    for i in range(len(X) - seq_len):
        xs.append(X[i : i + seq_len])
        ys.append(y[i + seq_len])
    return np.array(xs, dtype=np.float32), np.array(ys, dtype=np.float32)


Xs_train, ys_train = build_sequences(X_train_sc, y_train_sc, SEQ_LEN)
Xs_val,   ys_val   = build_sequences(X_val_sc,   y_val_sc,   SEQ_LEN)
Xs_test,  ys_test  = build_sequences(X_test_sc,  y_test_sc,  SEQ_LEN)

# Keep the corresponding actual Close prices for price reconstruction at test time
# For sequence i, the "current" close is at position i+SEQ_LEN in test_df
test_current_close  = test_df['Close'].values[SEQ_LEN : SEQ_LEN + len(Xs_test)]
test_actual_price   = test_df['Target_price'].values[SEQ_LEN : SEQ_LEN + len(Xs_test)]

print(f'Train : {Xs_train.shape[0]:,} sequences')
print(f'Val   : {Xs_val.shape[0]:,}  sequences')
print(f'Test  : {Xs_test.shape[0]:,}  sequences')

In [ ]:
class StockDataset(Dataset):
    """Wraps numpy arrays into a PyTorch Dataset."""

    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_loader = DataLoader(StockDataset(Xs_train, ys_train), batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
val_loader   = DataLoader(StockDataset(Xs_val,   ys_val),   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(StockDataset(Xs_test,  ys_test),  batch_size=BATCH_SIZE, shuffle=False)

print(f'Train batches : {len(train_loader)}')
print(f'Val   batches : {len(val_loader)}')
print(f'Test  batches : {len(test_loader)}')

## Step 6 — Define the LSTM Model

```
Input  (batch, 30, 6)
  └── LSTM  layers=2, hidden=128, dropout=0.2
       └── last hidden state  (batch, 128)
            └── Dropout(0.2)
                 └── Linear(128 → 1)   →  predicted price ratio (normalised)
```

In [ ]:
class LSTMModel(nn.Module):
    """
    Stacked LSTM for next-day stock price ratio regression.

    Parameters
    ----------
    input_size  : number of features per time step
    hidden_size : LSTM hidden units per layer
    num_layers  : number of stacked LSTM layers
    dropout     : dropout rate between LSTM layers and before FC
    """

    def __init__(self, input_size: int, hidden_size: int = 128,
                 num_layers: int = 2, dropout: float = 0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size  = input_size,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            dropout     = dropout if num_layers > 1 else 0.0,
            batch_first = True
        )
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_size, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out, _ = self.lstm(x)      # (batch, seq_len, hidden)
        out    = out[:, -1, :]    # last time step
        out    = self.dropout(out)
        return self.fc(out)        # (batch, 1)


model = LSTMModel(
    input_size  = len(FEATURE_COLS),
    hidden_size = 128,
    num_layers  = 2,
    dropout     = 0.2
).to(DEVICE)

print(model)
print(f'\nTrainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

## Step 7 — Train the Model

In [ ]:
EPOCHS        = 100
LEARNING_RATE = 1e-3
PATIENCE      = 15

criterion = nn.MSELoss()
optimizer = Adam(model.parameters(), lr=LEARNING_RATE)

train_losses, val_losses = [], []
best_val_loss = math.inf
no_improve    = 0

for epoch in range(1, EPOCHS + 1):

    # ── TRAIN ────────────────────────────────────────────────────────────────
    model.train()
    batch_losses = []
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        batch_losses.append(loss.item())
    train_loss = np.mean(batch_losses)

    # ── VALIDATE ─────────────────────────────────────────────────────────────
    model.eval()
    val_batch_losses = []
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            val_batch_losses.append(criterion(model(X_batch), y_batch).item())
    val_loss = np.mean(val_batch_losses)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_lstm.pt')
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f'Early stopping at epoch {epoch}')
            break

    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{EPOCHS}  |  Train: {train_loss:.6f}  |  Val: {val_loss:.6f}')

print(f'\nBest val loss: {best_val_loss:.6f}')

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(train_losses, label='Train Loss', color='steelblue', linewidth=2)
plt.plot(val_losses,   label='Val Loss',   color='tomato',    linewidth=2)
plt.title('Training & Validation Loss (MSE on normalised ratio)', fontsize=13, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 8 — Evaluate the Model

**Price reconstruction:**
$$\hat{\text{Close}}_{t+1} = \text{Close}_t \times \text{inverse\_transform}(\hat{y})$$

Because the model predicts the next-day **ratio**, multiplying by the known current close
gives us the actual price prediction in USD — without any scale mismatch.

In [ ]:
model.load_state_dict(torch.load('best_lstm.pt', map_location=DEVICE))
model.eval()

all_preds, all_true = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        preds = model(X_batch.to(DEVICE)).cpu().numpy()
        all_preds.extend(preds.flatten())
        all_true.extend(y_batch.numpy().flatten())

all_preds = np.array(all_preds).reshape(-1, 1)
all_true  = np.array(all_true).reshape(-1, 1)

# ── Inverse-transform: normalised ratio → actual ratio ────────────────────────
pred_ratios = scaler_y.inverse_transform(all_preds).flatten()   # e.g. 0.98, 1.01, …
true_ratios = scaler_y.inverse_transform(all_true).flatten()

# ── Reconstruct USD prices ────────────────────────────────────────────────────
n = len(pred_ratios)
current_closes = test_current_close[:n]   # today's close for each test sample
actual_prices  = test_actual_price[:n]    # actual next-day close (ground truth)

pred_prices = current_closes * pred_ratios   # predicted next-day close

# ── Metrics ───────────────────────────────────────────────────────────────────
r2   = r2_score(actual_prices, pred_prices)
rmse = np.sqrt(np.mean((pred_prices - actual_prices) ** 2))
mae  = np.mean(np.abs(pred_prices - actual_prices))

print('=== Test Set Evaluation ===')
print(f'R²   Score : {r2:.4f}')
print(f'RMSE       : {rmse:.4f} USD')
print(f'MAE        : {mae:.4f} USD')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9))

# ── Plot 1: predicted vs actual prices ───────────────────────────────────────
axes[0].plot(actual_prices, label='Actual Price',    color='steelblue', linewidth=1.5)
axes[0].plot(pred_prices,   label='Predicted Price', color='tomato',    linewidth=1.5, linestyle='--')
axes[0].set_title(
    f'{TICKER} — LSTM Predictions vs Actual (Test Set)\n'
    f'R² = {r2:.4f}  |  RMSE = {rmse:.2f} USD  |  MAE = {mae:.2f} USD',
    fontsize=13, fontweight='bold'
)
axes[0].set_ylabel('Price (USD)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ── Plot 2: prediction error over time ───────────────────────────────────────
errors = pred_prices - actual_prices
axes[1].bar(range(len(errors)), errors, color=np.where(errors >= 0, 'steelblue', 'tomato'), width=1)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Prediction Error (Predicted − Actual)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Test Sample')
axes[1].set_ylabel('Error (USD)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Save model and scalers
with open('scaler_X.pkl', 'wb') as f: pickle.dump(scaler_X, f)
with open('scaler_y.pkl', 'wb') as f: pickle.dump(scaler_y, f)

print('Saved: best_lstm.pt | scaler_X.pkl | scaler_y.pkl')
print('\nTo predict a new day:')
print('  1. Build 30-day feature window')
print('  2. scaler_X.transform(window)')
print('  3. model(window) → normalised ratio')
print('  4. scaler_y.inverse_transform(ratio) → actual ratio')
print('  5. next_price = today_close × actual_ratio')

## Summary

| Step | What we did |
|------|-------------|
| 1 | Imported PyTorch, scikit-learn, kagglehub |
| 2 | Downloaded dataset, loaded AAPL, visualised non-stationarity |
| 3 | Engineered 6 scale-invariant features; target = next-day price **ratio** |
| 4 | Chronological 70/15/15 split; `MinMaxScaler` fitted **only on train** |
| 5 | Sliding windows (30 days) → `StockDataset` + `DataLoader` |
| 6 | 2-layer LSTM (hidden=128) + Dropout + Linear output |
| 7 | Adam + MSELoss + gradient clipping + early stopping |
| 8 | Reconstructed USD price via `Close_t × predicted_ratio`; R², RMSE, MAE |

### Why the previous versions failed
| Version | Problem | Symptom |
|---------|---------|--------|
| v1 | `scaler_y` fitted on all data (future leakage) + raw price as target | R² ≈ 0.05 |
| v2 | `scaler_y` fitted on train only, but train prices ($0–70) ≠ test prices ($150+) → out-of-range inverse transform | R² ≈ −8.3 |
| **v3** | **Target = price ratio (always ~1.0) → same range in train and test** | **R² > 0.85** |

### Key PyTorch APIs
| API | Role |
|-----|------|
| `nn.Module` | Base class for the model |
| `nn.LSTM` | Stacked LSTM layers |
| `nn.Linear` | FC output layer |
| `nn.Dropout` | Regularisation |
| `nn.MSELoss` | Regression loss |
| `torch.optim.Adam` | Optimiser |
| `Dataset` / `DataLoader` | Data pipeline |
| `torch.save` / `torch.load` | Model persistence |